# 02 - Asset Incident Intelligence

## Purpose

Create asset-level operational incident intelligence.

This notebook identifies which assets show repeated, elevated, or long-duration operational incidents.

## Expected output

`vattenfall_dev.analytics.gold_asset_incident_intelligence`

## Main metrics

- total incident count
- elevated incident count
- critical incident count
- total duration
- average duration
- last incident day
- asset risk category

## Main idea

This output moves from regional summaries to asset-level operational risk intelligence.

In [0]:
import sys
import importlib
from pyspark.sql import functions as F

repo_src_path = "/Workspace/Users/dirella@startsteps.org/vattenfall-week9-capstone-anna/src"

if repo_src_path not in sys.path:
    sys.path.append(repo_src_path)

import risk.risk_classification as risk_classification
importlib.reload(risk_classification)

from risk.risk_classification import add_asset_risk_category

In [0]:
catalog = "vattenfall_dev"

grid_table = f"{catalog}.refined.silver_grid_events"
asset_table = f"{catalog}.refined.silver_asset_reference"

target_table = f"{catalog}.analytics.gold_asset_incident_intelligence"

print("Grid source:", grid_table)
print("Asset source:", asset_table)
print("Target table:", target_table)

In [0]:
grid_df = spark.table(grid_table)
asset_df = spark.table(asset_table)

print("Grid rows:", grid_df.count())
print("Asset rows:", asset_df.count())

display(grid_df.limit(20))
display(asset_df.limit(20))

In [0]:
grid_asset_df = (
    grid_df.alias("g")
    .join(
        asset_df.alias("a"),
        on="asset_id",
        how="left"
    )
    .select(
        F.col("g.asset_id"),
        F.col("a.asset_name"),
        F.col("g.region"),
        F.col("a.asset_type"),
        F.col("g.event_id"),
        F.col("g.event_day"),
        F.col("g.severity"),
        F.col("g.severity_band"),
        F.col("g.duration_minutes"),
    )
)

display(grid_asset_df.limit(20))

In [0]:
asset_incident_df = (
    grid_asset_df
    .groupBy(
        "asset_id",
        "asset_name",
        "region",
        "asset_type"
    )
    .agg(
        F.count("*").alias("total_incident_count"),
        F.sum(
            F.when(F.col("severity_band") == "ELEVATED", 1).otherwise(0)
        ).alias("elevated_incident_count"),
        F.sum(
            F.when(F.col("severity") == "CRITICAL", 1).otherwise(0)
        ).alias("critical_incident_count"),
        F.sum("duration_minutes").alias("total_duration_minutes"),
        F.round(F.avg("duration_minutes"), 2).alias("avg_duration_minutes"),
        F.max("event_day").alias("last_incident_day"),
    )
    .transform(add_asset_risk_category)
)

display(asset_incident_df)

In [0]:
(
    asset_incident_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

print("Written Night Shift asset incident intelligence table:", target_table)
print("Rows written:", asset_incident_df.count())

In [0]:
result_df = spark.table(target_table)

print("Rows in asset incident intelligence:", result_df.count())
print("Columns:", result_df.columns)

display(result_df)

print("Asset risk category distribution:")
result_df.groupBy("asset_risk_category").count().show()

In [0]:
duplicate_asset_rows = (
    result_df
    .groupBy("asset_id")
    .count()
    .filter("count > 1")
    .count()
)

print("Duplicate asset_id rows:", duplicate_asset_rows)

if duplicate_asset_rows > 0:
    raise ValueError("Asset incident intelligence validation failed: duplicate asset_id rows found.")

print("Asset incident intelligence validation passed.")